# DronAI · Difix3D+ en Kaggle

Entrena una escena de DronAI con **gsplat** dos veces en paralelo, una en cada T4:

- **base**: Gaussian splatting normal, 60.000 pasos.
- **difix**: **Difix3D+** (NVIDIA, CVPR 2025) adaptado a la web. Los mismos primeros 30.000 pasos y, en los otros
  30.000, cada 2.000 pasos dibuja vistas nuevas, les quita las manchas con un modelo de difusión y las suma al
  entrenamiento. La receta original apunta esas vistas a los cuadros apartados, que están sobre el recorrido del dron;
  aquí avanzan hacia dos anillos de vistas más altas y lejanas, como las del visor web, que es donde se ven las fallas.

Las dos corridas apartan 1 de cada 8 cuadros para medir (PSNR, SSIM y LPIPS), igual que las pruebas con Brush. Las vistas
de lejos no tienen foto real: se comparan a ojo, con imágenes de los dos modelos desde los mismos puntos. Al final todo se
exporta como PLY para la web.

**Antes de correr:**

1. *Settings → Accelerator*: **GPU T4 x2**.
2. *Settings → Internet*: **On** (para instalar gsplat y bajar el modelo de Hugging Face).
3. *Add Input*: el dataset con la escena, hecho con `dronai-worker kaggle-pack` (una carpeta `<escena>/` con `images/` y
   `sparse/0/`).

Después: *Run All* (o *Save Version → Save & Run All* para no dejar la pestaña abierta). Tarda entre 2 y 4 horas. Antes de entrenar, una celda de verificación compila gsplat y prueba
Difix con dos fotos de la escena; si algo falla se detiene ahí, sin gastar la hora de GPU. Los resultados quedan en
`/kaggle/working/resultados.zip`.

In [ ]:
SCENE = "kallang"    # nombre de la carpeta de la escena dentro del dataset
TEST_EVERY = 8       # 1 de cada 8 cuadros se aparta para medir, como en las pruebas con Brush
# Receta de NVIDIA: 30.000 pasos de entrenamiento normal y otros 30.000 en los que Difix arregla vistas nuevas cada
# 2.000 pasos. La corrida base hace los mismos 60.000 pasos sin Difix, para comparar a igual cantidad de pasos.
FIX_STEPS = list(range(30_000, 60_001, 2_000))
# Hacia dónde avanzan las vistas que arregla Difix: dos anillos de 12 vistas que miran al centro de la escena, más altos
# y lejanos que el dron (que vuela a unos 19 del centro y 6 de altura). El alto coincide con la vista inicial del visor
# web (45 de distancia, 40° de altura). Las vistas salen de las cámaras del dron y avanzan DIFIX_STEP por pasada: según
# la simulación, el anillo bajo se alcanza en la pasada 7 y el alto en la 14, de 16.
RINGS = [(20, 35, 12), (40, 45, 12)]  # (elevación en grados, distancia al centro, cantidad de vistas)
DIFIX_STEP = 2.0
DIFIX_SIDE = 1024  # lado mayor de las imágenes que procesa Difix: a 1598 px en fp32 no cabe en una T4
WORK = "/kaggle/working"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nvcc --version | tail -n 2
import torch
print("torch", torch.__version__, "· CUDA", torch.version.cuda, "· GPUs:", torch.cuda.device_count())
assert torch.cuda.device_count() >= 2, "Elige 'GPU T4 x2' en Settings → Accelerator"

In [ ]:
%%bash
set -e
cd /kaggle/working
[ -d Difix3D ] || git clone --depth 1 https://github.com/nv-tlabs/Difix3D.git
# Las carpetas de código de Difix3D no traen __init__.py, y sin él Python prefiere cualquier librería instalada con el
# mismo nombre: la "datasets" de Hugging Face que trae Kaggle tapaba datasets.colmap.
touch Difix3D/src/__init__.py Difix3D/examples/__init__.py Difix3D/examples/gsplat/datasets/__init__.py
# gsplat se compila para la GPU la primera vez que se usa (se hace en la celda de verificación).
pip install -q gsplat ninja
# Dependencias del ejemplo de gsplat (examples/gsplat/requirements.txt) menos "numpy<2": Kaggle usa Python 3.13 y jax,
# que necesitan numpy 2, y el lector de COLMAP y el código de Difix3D no usan nada que numpy 2 haya quitado.
pip install -q "git+https://github.com/rmbrualla/pycolmap@cc7ea4b7301720ac29287dbe450952511b32125e" \
  viser "nerfview==0.0.2" "imageio[ffmpeg]" scikit-learn tqdm "torchmetrics[image]" opencv-python \
  "tyro>=0.8.8" tensorboard tensorly pyyaml matplotlib
pip install -q --no-build-isolation "git+https://github.com/rahul-goel/fused-ssim@1272e21a282342e89537159e4bad508b19b34157"
# Difix: las versiones que fija su requirements.txt. Sin xformers, que obligaría a cambiar el torch de Kaggle.
pip install -q einops lpips "peft==0.9.0" "diffusers==0.25.1" "huggingface-hub==0.25.1" "transformers==4.38.0"
# Por si una corrida anterior dejó numpy 1.x instalado.
pip install -q "numpy>=2.1"
echo "Instalación lista"

In [ ]:
# Ajustes al código instalado. Se pueden volver a correr sin problema: cada uno se aplica una sola vez.
import importlib.util

def patch(path, old, new):
    text = open(path).read()
    if new not in text:
        assert old in text, f"No encontré lo que había que ajustar en {path}"
        open(path, "w").write(text.replace(old, new, 1))

TRAINER = f"{WORK}/Difix3D/examples/gsplat/simple_trainer_difix3d.py"

# 1. El lector de COLMAP de gsplat (pycolmap de rmbrualla) escribe "sin punto 3D" como np.uint64(-1), que numpy 2 ya
#    no acepta. Va el mismo número escrito de forma válida: el máximo de uint64.
pycolmap_dir = importlib.util.find_spec("pycolmap").submodule_search_locations[0]
patch(f"{pycolmap_dir}/scene_manager.py", "np.uint64(-1)", "np.uint64(np.iinfo(np.uint64).max)")

# 2. Con test_every=8, el Dataset de Difix3D entrena con 1 de cada 8 cuadros y mide con los otros 7 (su benchmark es de
#    vistas escasas). Lo dejamos como en gsplat y en Brush: entrena con 7 de cada 8 y aparta 1 de cada 8 para medir.
SPLIT = ("self.indices = indices[indices % self.parser.test_every {} 0]\n            else:\n"
         "                self.indices = indices[indices % self.parser.test_every {} 0]")
patch(f"{WORK}/Difix3D/examples/gsplat/datasets/colmap.py", SPLIT.format("==", "!="), SPLIT.format("!=", "=="))

# 3. Cada pasada de Difix deja vivo el cargador de datos anterior, con 4 procesos, aunque solo se usa el último: con 16
#    pasadas serían 64 procesos y la RAM de Kaggle podría no alcanzar. Se conserva solo el último; el método no cambia.
patch(TRAINER,
      "self.novelloaders.append(dataloader)\n        self.novelloaders_iter.append(iter(dataloader))",
      "self.novelloaders = [dataloader]\n        self.novelloaders_iter = [iter(dataloader)]")

# 4. Difix en fp16. En fp32 y a 1598 px pidió más de 15 GB y no cabe en una T4; en fp16 ocupa la mitad y va más rápido.
patch(TRAINER, 'DifixPipeline.from_pretrained("nvidia/difix_ref", trust_remote_code=True)',
      'DifixPipeline.from_pretrained("nvidia/difix_ref", trust_remote_code=True, torch_dtype=torch.float16)')

# 5. Lo que define este notebook (celda de objetivos), por variables de entorno: hacia qué vistas avanza Difix, cuánto en
#    cada pasada y a qué tamaño procesa las imágenes (después se escalan al tamaño original, como ya hacía el código).
#    Sin esas variables, el entrenador hace lo mismo que el original.
patch(TRAINER, "from src.pipeline_difix import DifixPipeline\n",
      "from src.pipeline_difix import DifixPipeline\n"
      "DIFIX_TARGETS = np.load(os.environ['DIFIX_TARGETS']) if os.environ.get('DIFIX_TARGETS') else None  # DronAI\n"
      "DIFIX_STEP = float(os.environ.get('DIFIX_STEP', '0.5'))  # DronAI\n"
      "DIFIX_SIZE = tuple(int(v) for v in os.environ['DIFIX_SIZE'].split('x')) if os.environ.get('DIFIX_SIZE') else None  # DronAI\n")
patch(TRAINER, "self.parser.camtoworlds[self.valset.indices], distance=0.5)",
      "self.parser.camtoworlds[self.valset.indices] if DIFIX_TARGETS is None else DIFIX_TARGETS, distance=DIFIX_STEP)")
patch(TRAINER, "image=image, ref_image=ref_image, num_inference_steps=1",
      "image=image.resize(DIFIX_SIZE, Image.LANCZOS) if DIFIX_SIZE else image, "
      "ref_image=ref_image.resize(DIFIX_SIZE, Image.LANCZOS) if DIFIX_SIZE else ref_image, num_inference_steps=1")
print("Ajustes listos")

In [ ]:
import glob, os

hits = sorted({os.path.dirname(os.path.dirname(os.path.dirname(p)))
               for p in glob.glob(f"/kaggle/input/**/{SCENE}/sparse/0/cameras.bin", recursive=True)})
assert hits, f"No encontré '{SCENE}/sparse/0' en /kaggle/input: agrega el dataset hecho con dronai-worker kaggle-pack"
DATA_DIR = hits[0]
print(DATA_DIR, "·", len(os.listdir(f"{DATA_DIR}/images")), "imágenes")

In [ ]:
# Vistas objetivo de Difix (ver RINGS): anillos de cámaras alrededor de la escena que miran a su centro, el punto al que
# miran todas las cámaras del dron (la alineación de DronAI lo deja en el origen, con z hacia arriba). Convención de
# COLMAP y OpenCV: x a la derecha, y hacia abajo, z hacia adelante.
import numpy as np
from PIL import Image

def look_at(eye, target=np.zeros(3), up=np.array([0.0, 0.0, 1.0])):
    forward = (target - eye) / np.linalg.norm(target - eye)
    right = np.cross(forward, up)
    right /= np.linalg.norm(right)
    pose = np.eye(4)
    pose[:3, :3] = np.stack([right, np.cross(forward, right), forward], axis=1)
    pose[:3, 3] = eye
    return pose

def ring_pose(elevation, distance, azimuth):
    e, a = np.radians(elevation), np.radians(azimuth)
    return look_at(distance * np.array([np.cos(a) * np.cos(e), np.sin(a) * np.cos(e), np.sin(e)]))

targets = np.array([ring_pose(e, d, 360 * k / n) for e, d, n in RINGS for k in range(n)])
TARGETS = f"{WORK}/difix_objetivos.npy"
np.save(TARGETS, targets)

w, h = Image.open(sorted(glob.glob(f"{DATA_DIR}/images/*"))[0]).size
scale = DIFIX_SIDE / max(w, h)
DIFIX_SIZE = f"{round(w * scale / 8) * 8}x{round(h * scale / 8) * 8}"
print(f"{len(targets)} vistas objetivo en {len(RINGS)} anillos · fotos de {w}x{h} · Difix trabaja a {DIFIX_SIZE}")

In [ ]:
# Verificación antes de entrenar. Cada prueba corre en un proceso aparte con el mismo entorno que el entrenamiento, y
# corren todas aunque alguna falle, para ver todos los problemas de una vez.
import subprocess, time

REPO = f"{WORK}/Difix3D"
ENV = {**os.environ, "PYTHONPATH": REPO,
       "USE_TF": "0", "USE_FLAX": "0",  # transformers importa TensorFlow y jax si están instalados; aquí sobran
       "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",  # menos memoria de video perdida en fragmentos
       # Para el ajuste 5: hacia qué vistas avanza Difix, cuánto por pasada y a qué tamaño trabaja.
       "DIFIX_TARGETS": TARGETS, "DIFIX_STEP": str(DIFIX_STEP), "DIFIX_SIZE": DIFIX_SIZE}

def trainer_args(out, extra=()):
    # Las opciones de las dos corridas, en un solo lugar: la verificación revisa que el entrenador las reconozca.
    return ["examples/gsplat/simple_trainer_difix3d.py", "default",
            "--data_dir", DATA_DIR, "--data_factor", "1", "--result_dir", out,
            "--no-normalize-world-space",  # conserva la alineación de DronAI (vertical en z)
            "--test_every", str(TEST_EVERY),
            "--save_steps", "30000", "60000",  # checkpoints solo a mitad y al final: cada uno pesa cientos de MB
            *extra]

# Prueba cada import del entrenador por separado y lista todos los que fallen, no solo el primero.
IMPORTS_TEST = '''
import ast, sys, traceback
sys.path.insert(0, "examples/gsplat")  # el entrenador corre como script desde su carpeta
failed = 0
for node in ast.parse(open("examples/gsplat/simple_trainer_difix3d.py").read()).body:
    if isinstance(node, (ast.Import, ast.ImportFrom)):
        try:
            exec(ast.unparse(node), {})
        except Exception:
            failed += 1
            print("✗", ast.unparse(node))
            for line in traceback.format_exc().strip().splitlines()[-3:]:
                print("     ", line)
print(f"fallaron {failed} imports" if failed else "todos los imports del entrenador funcionan")
sys.exit(1 if failed else 0)
'''

# Abre la escena con el lector de NVIDIA, como al arrancar el entrenamiento, y carga una foto.
PARSER_TEST = f'''
import sys
sys.path.insert(0, "examples/gsplat")
from datasets.colmap import Dataset, Parser
parser = Parser(data_dir="{DATA_DIR}", factor=1, normalize=False, test_every={TEST_EVERY})
train, val = Dataset(parser, split="train"), Dataset(parser, split="val")
h, w = train[0]["image"].shape[:2]
print(f"{{len(train)}} cuadros para entrenar y {{len(val)}} para medir, de {{w}}x{{h}} · {{len(parser.points):,}} puntos iniciales")
'''

RASTER_TEST = '''
import torch, gsplat
d = "cuda"
means = torch.zeros(1, 3, device=d); quats = torch.tensor([[1.0, 0, 0, 0]], device=d)
scales = torch.full((1, 3), 0.1, device=d); opac = torch.ones(1, device=d); colors = torch.ones(1, 3, device=d)
view = torch.eye(4, device=d)[None]; view[0, 2, 3] = 3.0
K = torch.tensor([[[50.0, 0, 32], [0, 50.0, 32], [0, 0, 1]]], device=d)
img, alpha, _ = gsplat.rasterization(means, quats, scales, opac, colors, view, K, 64, 64)
print("gsplat", gsplat.__version__, "dibuja; opacidad en el centro:", round(float(alpha[0, 32, 32, 0]), 2))
'''

# Difix como lo usará el entrenador ya ajustado: en fp16 y a DIFIX_SIZE, con una foto y su vecina como referencia.
VIEWS = len(targets)  # vistas que arregla en cada pasada
DIFIX_TEST = f'''
import glob, time, numpy as np, torch
from PIL import Image
from src.pipeline_difix import DifixPipeline
pipe = DifixPipeline.from_pretrained("nvidia/difix_ref", trust_remote_code=True, torch_dtype=torch.float16)
pipe.set_progress_bar_config(disable=True)
pipe.to("cuda")
model_gb = torch.cuda.memory_allocated() / 2**30
size = tuple(int(v) for v in "{DIFIX_SIZE}".split("x"))
image, ref = (Image.open(p).convert("RGB").resize(size, Image.LANCZOS) for p in sorted(glob.glob("{DATA_DIR}/images/*"))[:2])
for _ in range(2):  # la primera vuelta calienta la GPU
    torch.cuda.synchronize(); t = time.time()
    out = pipe(prompt="remove degradation", image=image, ref_image=ref, num_inference_steps=1, timesteps=[199],
               guidance_scale=0.0).images[0]
    torch.cuda.synchronize(); dt = time.time() - t
peak_gb = torch.cuda.max_memory_allocated() / 2**30
pixels = np.asarray(out, dtype=np.float32)
assert np.isfinite(pixels).all() and pixels.std() > 5, "Difix devolvió una imagen vacía en fp16"
print(f"{{size[0]}}x{{size[1]}} en fp16: {{dt:.1f}} s por vista → {{{len(FIX_STEPS) * VIEWS} * dt / 60:.0f}} min en las "
      f"{len(FIX_STEPS)} pasadas de {VIEWS} vistas · VRAM: {{model_gb:.1f}} GB el modelo, {{peak_gb:.1f}} GB en el pico "
      f"(más lo que ocupe gsplat)")
'''

def check(title, args, gpu=None, expect=()):
    # Corre una prueba y muestra su última línea, o el error completo si falla. expect: textos que deben salir.
    print(title, "…", end=" ", flush=True)
    t = time.time()
    env = ENV if gpu is None else {**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)}
    r = subprocess.run(["python", *args], cwd=REPO, env=env, capture_output=True, text=True)
    missing = [e for e in expect if e not in r.stdout.replace("_", "-")]
    ok = r.returncode == 0 and not missing
    print(f"{'OK' if ok else 'FALLÓ'} ({time.time() - t:.0f} s)")
    if r.returncode:
        print((r.stdout + r.stderr)[-3000:])
    elif missing:
        print("    el entrenador no reconoce:", ", ".join(missing))
    elif not expect:
        print("   ", (r.stdout.strip().splitlines() or [""])[-1])
    return ok

FLAGS = [a.replace("_", "-") for a in trainer_args("x", ["--fix_steps", "1"]) if a.startswith("--")]
ok = {"imports": check("Imports del entrenador", ["-c", IMPORTS_TEST])}
ok["opciones"] = ok["imports"] and check("Opciones del entrenador", [*trainer_args("x")[:2], "--help"], expect=FLAGS)
ok["datos"] = check("El lector de NVIDIA abre la escena", ["-c", PARSER_TEST])
ok["gsplat"] = check("gsplat compila y dibuja (la primera vez tarda varios minutos)", ["-c", RASTER_TEST])
ok["difix"] = check("Difix baja el modelo y arregla una foto de la escena", ["-c", DIFIX_TEST], gpu=0)
assert all(ok.values()), "Algo falló: mándame todo el texto de esta celda"

In [ ]:
def launch(name, gpu, extra):
    out = f"{WORK}/runs/{SCENE}-{name}"
    os.makedirs(out, exist_ok=True)
    env = {**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)}
    log = open(f"{out}/log.txt", "w")
    return subprocess.Popen(["python", *trainer_args(out, extra)], cwd=REPO, env=env,
                            stdout=log, stderr=subprocess.STDOUT), out

# La base nunca llega a una pasada de Difix; la otra sigue la receta de NVIDIA (FIX_STEPS).
runs = {"base": launch("base", 0, ["--fix_steps", "100000000"]),
        "difix": launch("difix", 1, ["--fix_steps", *map(str, FIX_STEPS)])}
start = time.time()
print("Entrenando en las dos T4:", ", ".join(runs))

In [ ]:
import psutil
from IPython.display import clear_output

def last_line(path):
    with open(path, errors="ignore") as f:
        lines = [l for l in f.read().replace("\r", "\n").splitlines() if l.strip()]
    return lines[-1][-170:] if lines else "(sin salida todavía)"

def memory():
    # RAM de la máquina y memoria de video de cada T4 (la 0 entrena la base, la 1 la de Difix), en GB.
    gpus = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                          capture_output=True, text=True).stdout.split()
    return {"RAM": psutil.virtual_memory().used / 2**30, **{f"T4 {n}": int(g) / 1024 for n, g in zip(runs, gpus)}}

peak = {}
while any(p.poll() is None for p, _ in runs.values()):
    now = memory()
    peak = {k: max(v, peak.get(k, 0)) for k, v in now.items()}
    clear_output(wait=True)
    print(f"{(time.time() - start) / 60:.0f} min · " + " · ".join(f"{k} {v:.1f} GB" for k, v in now.items()))
    for name, (p, out) in runs.items():
        print(f"{name:>5} | {'listo' if p.poll() is not None else 'corriendo'} | {last_line(out + '/log.txt')}")
    time.sleep(60)
clear_output(wait=True)
print(f"Terminó en {(time.time() - start) / 60:.0f} min ·", {n: p.returncode for n, (p, _) in runs.items()})
print("Máximos:", " · ".join(f"{k} {v:.1f} GB" for k, v in peak.items()),
      f"(RAM total {psutil.virtual_memory().total / 2**30:.0f} GB)")
for name, (p, out) in runs.items():
    if p.returncode != 0:
        print(f"\n--- {name} falló; últimas líneas de {out}/log.txt ---")
        !tail -n 40 {out}/log.txt

In [ ]:
import json, re

def stats(out):
    rows = {}
    for f in glob.glob(f"{out}/stats/val_step*.json"):
        rows[int(re.search(r"val_step(\d+)", f).group(1)) + 1] = json.load(open(f))
    return rows

table = {name: stats(out) for name, (_, out) in runs.items()}
print(f"{'paso':>7} │ {'PSNR base':>9} {'difix':>7} │ {'SSIM base':>9} {'difix':>7} │ {'LPIPS base':>10} {'difix':>7}")
for step in sorted(set(table["base"]) | set(table["difix"])):
    cell = lambda name, key: f"{table[name][step][key]:.3f}" if step in table[name] and key in table[name][step] else "–"
    print(f"{step:>7} │ {cell('base','psnr'):>9} {cell('difix','psnr'):>7} │ {cell('base','ssim'):>9} {cell('difix','ssim'):>7}"
          f" │ {cell('base','lpips'):>10} {cell('difix','lpips'):>7}")
os.makedirs(f"{WORK}/resultados", exist_ok=True)
json.dump(table, open(f"{WORK}/resultados/{SCENE}-metricas.json", "w"), indent=1)
print("\nPSNR y SSIM: más es mejor. LPIPS: menos es mejor.")

In [ ]:
import numpy as np

def ckpt_to_ply(ckpt_path, ply_path):
    # Del checkpoint de gsplat al PLY estándar de 3DGS (el que leen Brush, Spark y el conversor a SPZ de DronAI).
    s = torch.load(ckpt_path, map_location="cpu", weights_only=False)["splats"]
    means, scales, quats = s["means"].numpy(), s["scales"].numpy(), s["quats"].numpy()
    opacities, sh0, shN = s["opacities"].numpy(), s["sh0"].numpy(), s["shN"].numpy()
    n = len(means)
    f_dc = sh0.reshape(n, 3)
    f_rest = shN.transpose(0, 2, 1).reshape(n, -1)  # [N, K-1, 3] → canales primero, como en 3DGS
    names = (["x", "y", "z", "nx", "ny", "nz"] + [f"f_dc_{i}" for i in range(3)]
             + [f"f_rest_{i}" for i in range(f_rest.shape[1])] + ["opacity"]
             + [f"scale_{i}" for i in range(3)] + [f"rot_{i}" for i in range(4)])
    data = np.concatenate([means, np.zeros_like(means), f_dc, f_rest, opacities.reshape(n, 1), scales, quats], axis=1)
    header = f"ply\nformat binary_little_endian 1.0\nelement vertex {n}\n"
    header += "".join(f"property float {name}\n" for name in names) + "end_header\n"
    with open(ply_path, "wb") as f:
        f.write(header.encode("ascii"))
        f.write(data.astype("<f4").tobytes())
    return n

for name, (_, out) in runs.items():
    ckpts = sorted(glob.glob(f"{out}/ckpts/ckpt_*_rank0.pt"), key=lambda p: int(re.search(r"ckpt_(\d+)_", p).group(1)))
    if not ckpts:
        print(name, "no dejó checkpoints"); continue
    n = ckpt_to_ply(ckpts[-1], f"{WORK}/resultados/{SCENE}-{name}.ply")
    print(f"{name}: {os.path.basename(ckpts[-1])} → {SCENE}-{name}.ply ({n:,} splats)")

In [ ]:
# Comparación visual. El entrenador guarda renders/val/<paso>/{GT,Pred}/NNNN.png en cada validación y
# renders/novel/<paso>/{Pred,Fixed}/NNNN.png en cada pasada de Difix.
from PIL import Image

def last_dir(out, kind):
    root = f"{out}/renders/{kind}"
    steps = [int(d.name) for d in os.scandir(root) if d.name.isdigit()] if os.path.isdir(root) else []
    return f"{root}/{max(steps)}" if steps else None

def strip(paths, width=960):
    tiles = [Image.open(p).convert("RGB") for p in paths]
    tiles = [t.resize((width, round(t.height * width / t.width)), Image.LANCZOS) for t in tiles]
    canvas = Image.new("RGB", (width * len(tiles), tiles[0].height))
    for i, tile in enumerate(tiles):
        canvas.paste(tile, (i * width, 0))
    return canvas

base_val, difix_val = last_dir(runs["base"][1], "val"), last_dir(runs["difix"][1], "val")
if base_val and difix_val:
    for i in range(min(4, len(os.listdir(f"{base_val}/GT")))):
        n = f"{i:04d}.png"
        strip([f"{base_val}/GT/{n}", f"{base_val}/Pred/{n}", f"{difix_val}/Pred/{n}"]).save(
            f"{WORK}/resultados/{SCENE}-validacion-{i}.jpg", quality=90)
    print("Cuadro apartado (el modelo nunca lo vio) · foto real | base | Difix3D+")
    display(Image.open(f"{WORK}/resultados/{SCENE}-validacion-0.jpg"))
else:
    print("No hay renders de validación")

novel = last_dir(runs["difix"][1], "novel")
if novel:
    for i in range(min(2, len(os.listdir(f"{novel}/Fixed")))):
        strip([f"{novel}/Pred/{i:04d}.png", f"{novel}/Fixed/{i:04d}.png"]).save(
            f"{WORK}/resultados/{SCENE}-difix-{i}.jpg", quality=90)
    print("Vista nueva en la última pasada de Difix · render de los splats | lo que Difix corrigió")
    display(Image.open(f"{WORK}/resultados/{SCENE}-difix-0.jpg"))

In [ ]:
# Las vistas que más importan para la web, desde lejos y desde arriba, no tienen foto real: se comparan a ojo. Dibuja los
# dos modelos finales desde la posición inicial del visor web y desde cuatro vistas del anillo alto, lado a lado.
SHOWCASE = {"visor": look_at(np.array([23.834, -23.834, 28.283]))}  # vista inicial del visor, en ejes de COLMAP
SHOWCASE.update({f"anillo-alto-{a}": ring_pose(*RINGS[-1][:2], a) for a in (0, 90, 180, 270)})
np.save(f"{WORK}/vistas_comparacion.npy", np.array(list(SHOWCASE.values())))
ckpts = {}
for name, (_, out) in runs.items():
    found = sorted(glob.glob(f"{out}/ckpts/ckpt_*_rank0.pt"), key=lambda p: int(re.search(r"ckpt_(\d+)_", p).group(1)))
    if found:
        ckpts[name] = found[-1]

RENDER_SCRIPT = f'''
import sys, numpy as np, torch
from PIL import Image
from gsplat import rasterization
sys.path.insert(0, "examples/gsplat")
from datasets.colmap import Parser
parser = Parser(data_dir="{DATA_DIR}", factor=1, normalize=False, test_every={TEST_EVERY})
K = torch.tensor(list(parser.Ks_dict.values())[0], dtype=torch.float32, device="cuda")[None]
w, h = list(parser.imsize_dict.values())[0]
poses = torch.tensor(np.load("{WORK}/vistas_comparacion.npy"), dtype=torch.float32, device="cuda")
tiles = {{}}
for model, path in {ckpts}.items():
    s = torch.load(path, map_location="cuda", weights_only=False)["splats"]
    colors = torch.cat([s["sh0"], s["shN"]], 1)
    for name, pose in zip({list(SHOWCASE)}, poses):
        image, _, _ = rasterization(s["means"], s["quats"], torch.exp(s["scales"]), torch.sigmoid(s["opacities"]),
                                    colors, torch.linalg.inv(pose)[None], K, w, h, sh_degree=3)
        tiles.setdefault(name, []).append((image[0].clamp(0, 1).cpu().numpy() * 255).astype(np.uint8))
    del s, colors
    torch.cuda.empty_cache()
for name, images in tiles.items():
    strip = Image.fromarray(np.concatenate(images, axis=1))
    strip = strip.resize((960 * len(images), round(strip.height * 960 * len(images) / strip.width)), Image.LANCZOS)
    strip.save("{WORK}/resultados/{SCENE}-lejos-" + name + ".jpg", quality=90)
print("vistas dibujadas:", ", ".join(tiles))
'''
r = subprocess.run(["python", "-c", RENDER_SCRIPT], cwd=REPO, env={**ENV, "CUDA_VISIBLE_DEVICES": "0"},
                   capture_output=True, text=True)
if r.returncode == 0:
    print(r.stdout.strip().splitlines()[-1])
    print("Desde la posición inicial del visor web · izquierda: base · derecha: Difix3D+")
    display(Image.open(f"{WORK}/resultados/{SCENE}-lejos-visor.jpg"))
else:
    print("No pude dibujar las vistas de lejos (lo demás sí quedó guardado):\n" + (r.stdout + r.stderr)[-2000:])

In [ ]:
!cd /kaggle/working && zip -qr resultados.zip resultados && ls -lh resultados.zip resultados
print("Bájalo desde la pestaña Output (resultados.zip) y en DronAI: node scripts/splats-to-spz.mjs ...")